<center>
  <h3>More than a Hack 2027 &middot; Phase 1</h3>
  <h2>Frame-level Ball Possession &mdash; 014 Bola Mati Terawasi di Decoder</h2>

  <p><strong>Metrica Sports &times; FC Barcelona Innovation Hub &times; Mobile World Capital</strong></p>
</center>

---

### Three Makasih

- **Anggota 1** (<span style="color:gray">email</span>)
- **Anggota 2** (<span style="color:gray">email</span>)
- **Anggota 3** (<span style="color:gray">email</span>)

---

## Ringkasan

Eksperimen 013 melatih TCN pada dua pertandingan Metrica lain dan menghasilkan peluang bola mati per frame untuk pertandingan ini. Kesesuaiannya dengan fakta luar sangat baik: porsi bola mati prediksi 42,9% pada seluruh frame dinilai, dibanding waktu bermain efektif La Liga 59,3% (Football Observatory, 37 kompetisi), yaitu sekitar 40,7% mati. Eksperimen ini memasukkan peluang itu ke decoder sebagai sumber label `D`, menggantikan klasifikasi gap 004 yang hanya mencakup sekitar seperempat henti permainan.

Perubahan dibuat terpisah: (1) sumber `D` dari 013 dengan batas peluang 0,5 yang tidak diubah dan durasi minimum 2 detik; (2) bukti tim bola dengan gerbang radius 1,0 m (rentang literatur 0,5 sampai 1,0 m) dan 1,5 m (pilihan sebelumnya), serta bukti bentuk tim dari 008 atau dari kepala tim 013; (3) λ dikalibrasi ulang ke laju possession Metrica seperti 012.

GT publik hanya dipakai untuk pemeriksaan pipeline. Batas 0,5 dan durasi minimum 2 detik ditetapkan sebelum melihat hasil dan tidak disetel pada GT.


# Metodologi

**peluang bola mati 013 (dilatih pada tiga game Metrica) → `D` bila p > 0,5 selama ≥ 2 detik → bukti tim (bola bergerbang + bentuk tim) → Viterbi H/A → λ dikalibrasi ke laju possession Metrica (012) → `out/*.csv` → validasi → skor GT publik (hanya pemeriksaan pipeline).**

| asumsi | nilai | alasan |
|---|---|---|
| batas bola mati | p(mati) > 0,5, tanpa disetel | probabilitas terkalibrasi pada prior Metrica; ditetapkan di awal |
| durasi minimum `D` | 2 detik | henti permainan nyata berlangsung beberapa detik; label Metrica bermedian 15 sampai 21 dtk |
| radius gerbang kendali | 1,0 m (literatur: 0,5 sampai 1,0 m) dan 1,5 m (011, 012) | zona penguasaan pada Vidal-Codina dkk. |
| bukti bentuk tim | κ = 0,02, dari 008 atau kepala tim 013 | sama dengan 007 sampai 012 |
| λ | dikalibrasi ke 344 ± 32 possession (Metrica) | 012 |

# Environment

Dijalankan dari dalam repositori. Peluang diambil dari cache 013 (`_cache/p013.npz`) dan 008 (`_cache/p_tcn.npy`); acuan laju possession dari data peristiwa Metrica (012). Tidak ada data pertandingan ini yang dikirim ke mana pun.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
for p in (ROOT, ROOT / "starter", ROOT / "evaluation", ROOT / "experiments"):
    sys.path.insert(0, str(p))

import subprocess
import warnings
warnings.filterwarnings("ignore")
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

import common
import other_games as og
import possession_metrics as pm
from baseline import smooth, write_submission
from games import GAME, GAMES, anchor_video_frame, half_video_blocks, source_frames

C1, C2, C3, INK, GREY = "#e94f37", "#393e41", "#f6f7eb", "#393e41", "#a3a7a6"
mpl.rcParams.update({
    "figure.facecolor": C3, "savefig.facecolor": C3, "axes.facecolor": "white",
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "legend.frameon": False,
    "figure.dpi": 110, "axes.prop_cycle": mpl.cycler(color=[C1, C2, GREY]),
})
def panel(ax, l): ax.text(-0.09, 1.07, l, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")

FPS = 25
N = source_frames("tactical")
BLOCKS3 = half_video_blocks("BAR-ATH")
BL = [(a, b) for _, a, b in BLOCKS3]
GRADED = np.zeros(N, bool)
for _, a, b in BLOCKS3: GRADED[a:b + 1] = True
GRADED_MIN = GRADED.sum() / FPS / 60
OUT = Path(common.HERE) / "014_supervised_dead_ball" / "out"; OUT.mkdir(parents=True, exist_ok=True)
KICKOFF2 = anchor_video_frame(GAMES[GAME]["kickoff_2nd"]) + 1
assert KICKOFF2 == 113742
MIN_D = 2 * FPS                                    # durasi minimum satu henti permainan: 2 detik (prior)
print(f"frames: {N} | dinilai: {GRADED.sum()} ({GRADED_MIN:.1f} menit)")

# Approach: Bola Mati dari Model Terawasi

## [1] Bola mati: seberapa masuk akal?

Dua pemeriksaan eksternal: porsi mati seluruh pertandingan terhadap waktu bermain efektif yang dipublikasikan (La Liga 59,3% berarti sekitar 40,7% mati), dan distribusi durasi henti permainan terhadap label Metrica (median 15 sampai 21 dtk).

In [ ]:
z = np.load(Path(common.CACHE) / "p013.npz")
PD, PT013 = z["pd_real"], z["pt_mask"]

def dead_mask(p, thr=0.5, min_len=MIN_D):
    m = np.zeros(N, bool)
    for _, a, b in BLOCKS3:
        seg = p[a:b + 1] > thr
        s, e = common.runs(seg)
        for si, ei in zip(s, e):
            if ei - si >= min_len: m[a + si:a + ei] = True
    return m

DEADM = dead_mask(PD)
s, e = common.runs(DEADM & GRADED); dur = (e - s) / FPS
print(f"porsi bola mati prediksi: {DEADM[GRADED].mean():.1%} frame dinilai (acuan terbit La Liga: 40,7%)")
print(f"henti permainan: {len(s)} | median {np.median(dur):.1f} dtk | rata-rata {dur.mean():.1f} dtk | per menit {len(s) / GRADED_MIN:.2f}")
ref, POSS_RATE = [], []
for n in (1, 2, 3):
    y = og.load(n)["y"]; s_, e_ = common.runs(y == 2); mins_ = np.sum(y >= 0) / FPS / 60
    ref.append(((e_ - s_) / FPS, len(s_) / mins_)); POSS_RATE.append(sum(len(common.runs(y == t)[0]) for t in (0, 1)) / mins_)
print("acuan Metrica (label peristiwa): median", [round(float(np.median(r[0])), 1) for r in ref], "dtk | henti per menit", [round(r[1], 2) for r in ref], "| possession per menit", [round(r, 2) for r in POSS_RATE])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(np.clip(dur, 0, 90), bins=30, color=C1, edgecolor=C3); ax[0].set(xlabel="durasi henti permainan prediksi (dtk, dipotong 90)", ylabel="jumlah", title="Pertandingan ini")
ax[1].hist(np.clip(np.concatenate([r[0] for r in ref]), 0, 90), bins=30, color=C2, edgecolor=C3); ax[1].set(xlabel="durasi henti permainan (dtk, dipotong 90)", ylabel="jumlah", title="Metrica (label peristiwa)")
panel(ax[0], "A"); panel(ax[1], "B"); fig.tight_layout(); plt.show()

Porsi mati prediksi mendekati angka terbit La Liga, dan distribusi durasi henti permainan memiliki bentuk yang mirip dengan label Metrica. Keduanya adalah korobrasi dari sumber yang tidak menyentuh ground truth tantangan.

## [2] Decoder dan kalibrasi

Bukti bola (jarak pemain terdekat, bergerbang radius *r*) dan bukti bentuk tim (κ = 0,02) menggerakkan Viterbi H/A; frame `D` menimpa hasilnya. λ dikalibrasi per varian ke target Metrica.

In [ ]:
dist = common.team_distances(); dH, dA = dist[:, 0], dist[:, 1]
OBS = np.isfinite(dH) & np.isfinite(dA)
NEAR = np.where(OBS, np.minimum(dH, dA), np.inf)
E = np.zeros(N); E[OBS] = np.clip((dA[OBS] - dH[OBS]) / 3.0, -10, 10)
LPB_H, LPB_A = -np.logaddexp(0, -E), -np.logaddexp(0, E); LPB_H[~OBS] = 0; LPB_A[~OBS] = 0
P008 = np.load(Path(common.CACHE) / "p_tcn.npy")
def logp(P):
    pc = np.clip(P, 1e-3, 1 - 1e-3); return np.log(pc), np.log(1 - pc)
LS = {"008": logp(P008), "013": logp(PT013)}

def finish(paths, dmask):
    lab = np.full(N, "X", dtype="<U1")
    for (half, a, b), p in zip(BLOCKS3, paths):
        lab[a:b + 1] = np.where(p == 0, "H", "A")
        if half == 2: lab[a:KICKOFF2] = "D"
    lab[dmask & GRADED] = "D"
    return lab

def decode(r, shape, lam, dmask=DEADM, ks=0.02):
    gate = NEAR <= r
    sh, sa = LS[shape]
    h = np.where(gate, LPB_H, 0.0) + ks * sh; a = np.where(gate, LPB_A, 0.0) + ks * sa
    return finish([common.viterbi_2state(h[x:y + 1], a[x:y + 1], lam) for x, y in BL], dmask)

def count(lab):
    n, gaps = 0, []
    for _, a, b in BLOCKS3:
        seg = lab[a:b + 1]; st = []
        for t in ("H", "A"):
            s_, e_ = common.runs(seg == t); st += list(s_)
        st = np.sort(st); n += len(st)
        if len(st) > 1: gaps += list(np.diff(st) / FPS)
    return n, np.array(gaps)

RATE_M = float(np.mean(POSS_RATE)); RATE_SD = max(float(np.std(POSS_RATE, ddof=1)), 0.2)      # tiga game; SD tidak diperkecil di bawah 0,2
T_MET, T_MET_SD = RATE_M * GRADED_MIN, RATE_SD * GRADED_MIN
print(f"target Metrica: {T_MET:.0f} +/- {T_MET_SD:.0f} possession")

SPEC = {"r10_008": (1.0, "008"), "r15_008": (1.5, "008"), "r10_013": (1.0, "013")}
LAMS = [2, 3, 5, 8, 12, 20, 30, 50, 80, 120]
SWEEP = {k: [(l, count(decode(r, sh, l))[0]) for l in LAMS] for k, (r, sh) in SPEC.items()}
def solve(sw, target):
    x = np.log([l for l, _ in sw]); y = np.array([n for _, n in sw], float)
    if target >= y[0]: return float(np.exp(x[0]))
    if target <= y[-1]: return float(np.exp(x[-1]))
    return float(np.exp(np.interp(-target, -y, x)))
CAL = {k: solve(v, T_MET) for k, v in SWEEP.items()}
display(pd.DataFrame({k: [n for _, n in v] for k, v in SWEEP.items()}, index=[f"λ={l}" for l in LAMS]).T)
display(pd.DataFrame({"λ* (Metrica)": CAL}).round(1))

Dengan `D` lebih banyak (sekitar 43% dibanding 22,7% pada 012), jumlah possession pada λ yang sama turun, karena henti permainan memutus possession dan frame `D` tidak lagi menjadi bagian dari run H atau A. Karena itu λ* dikalibrasi ulang, bukan dipindahkan dari 012.

## [3] Varian yang dibangun

In [ ]:
VARS = {"v1_r10_008": ("r10_008", 1.0, "008"), "v2_r15_008": ("r15_008", 1.5, "008"), "v3_r10_013": ("r10_013", 1.0, "013")}
for old in OUT.glob("*.csv"): old.unlink()
LAB, rows = {}, []
for name, (key, r_, sh) in VARS.items():
    lab = decode(r_, sh, CAL[key]); LAB[name] = lab; write_submission(str(OUT / f"{name}.csv"), lab)

# acuan: 012 (D dari klasifikasi gap 004), dibaca dari berkas 012
ref12 = pd.read_csv(ROOT / "experiments" / "012_metrica_recalibration" / "out" / "v1_metrica_gate15.csv")["label"].astype(str).values
LAB["ref_012_gapD"] = ref12; write_submission(str(OUT / "ref_012_gapD.csv"), ref12)

for name, lab in LAB.items():
    n, g = count(lab)
    rows.append({"varian": name, "λ": round(CAL[VARS[name][0]], 1) if name in VARS else 5.6, "possession": n, "per menit": round(n / GRADED_MIN, 2),
                 "median awal-ke-awal (dtk)": round(float(np.median(g)), 1), "% D": round(100 * np.mean(lab[GRADED] == "D"), 1)})
codes = common.cached("atd_nearest", lambda: None)
base = np.full(N, "X", dtype="<U1"); sm = smooth(codes, 50)
for _, a, b in BLOCKS3: base[a:b + 1] = sm[a:b + 1]
write_submission(str(OUT / "baseline_smooth50.csv"), base)
n, g = count(base)
rows.append({"varian": "baseline_smooth50", "λ": None, "possession": n, "per menit": round(n / GRADED_MIN, 2), "median awal-ke-awal (dtk)": round(float(np.median(g)), 1), "% D": round(100 * np.mean(base[GRADED] == "D"), 1)})
display(pd.DataFrame(rows))

## [4] Validasi, skor, dan diagnostik (pemeriksaan pipeline)

Skor GT publik hanya memeriksa pipeline; dengan 17 possession selisih kecil adalah noise. Karena porsi `D` pada jendela ini nyata (19,6%), pemeriksaan bola mati di sini informatif untuk **kelas D**, bukan untuk membandingkan varian tim.

In [ ]:
for f in sorted(OUT.glob("*.csv")):
    r = subprocess.run([sys.executable, str(ROOT / "evaluation" / "validate_submission.py"), str(f)], capture_output=True, text=True)
    print(f.name, "->", "VALID" if "Valid." in r.stdout else r.stdout[-200:])
subprocess.run([sys.executable, str(ROOT / "evaluation" / "run_evaluation.py"), "--results", str(OUT), "--out", str(OUT / "reports")], capture_output=True, text=True)
board = pd.read_csv(OUT / "reports" / "scoreboard.csv")
display(board[["rank", "submission", "macro_f1", "spell_f1_mean", "poss", "dur"]].round(4))

gt = pm.read_label_csv(ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv")["label"].astype(str).values
a0, b0 = 11185, 18684; w = np.arange(a0, b0 + 1)
gd = gt[w] == "D"
drows = []
for name, lab in LAB.items():
    pd_ = lab[w] == "D"
    drows.append({"varian": name, "D prediksi": f"{pd_.mean():.1%}", "D sebenarnya": f"{gd.mean():.1%}", "recall D": f"{pd_[gd].mean():.1%}", "presisi D": f"{gd[pd_].mean():.1%}" if pd_.any() else "-",
                  "akurasi tim (frame hidup bersama)": f"{np.mean((lab[w] == gt[w])[(gt[w] != 'D') & (lab[w] != 'D')]):.1%}"})
display(pd.DataFrame(drows))

# Ringkasan 014

**Yang dikerjakan.** Peluang bola mati dari TCN terawasi 013 (dilatih pada dua pertandingan Metrica lain, tidak pernah melihat pertandingan ini) menjadi sumber label `D`: `D` bila p > 0,5 selama minimal 2 detik, dua angka yang ditetapkan sebelum melihat hasil. Menggantikan klasifikasi gap 004. Bukti tim: jarak pemain terdekat bergerbang radius 1,0 m (rentang literatur) atau 1,5 m, plus bukti bentuk tim dari 008 atau dari kepala tim 013; λ dikalibrasi ulang ke laju possession Metrica (344 ± 32 possession). Semua empat berkas lolos `validate_submission.py`. Tidak ada label tangan pada pertandingan ini dan GT tidak dipakai untuk melatih atau memilih parameter.

**Kesesuaian dengan fakta eksternal.**
1. Porsi bola mati prediksi 42,6% frame dinilai, dibanding waktu bermain efektif La Liga yang diterbitkan 59,3% (sekitar 40,7% mati; Football Observatory, 37 kompetisi). Keyakinan lama saya bahwa porsi mati sekitar 20% berasal dari satu jendela 5 menit awal yang ternyata tidak representatif.
2. 103 henti permainan dengan median 20,2 detik, sebanding dengan label Metrica; 0,77 dan 0,82 henti per menit vs 0,97 di sini).

**Pemeriksaan pipeline pada GT publik (bukan dasar penyetelan).**

| | macroF1 | recall D | presisi D | akurasi tim (frame hidup bersama) | possession (sebenarnya 17) |
|---|---|---|---|---|---|
| `ref_012_gapD` (gap 004) | 0,667 | 24,9% | 100% | 87,3% | 15 |
| `v1_r10_008` | 0,833 | 77,9% | 85,2% | 88,3% | 18 |
| `v2_r15_008` | 0,846 | 77,9% | 85,2% | 90,2% | 14 |
| `v3_r10_013` | 0,857 | 77,9% | 85,2% | 92,0% | 19 |
| `baseline_smooth50` | 0,621 | | | | 26 |

Kenaikan macroF1 sekitar 0,17 hampir seluruhnya berasal dari kelas D (recall dari 25% menjadi 78%), bukan dari keputusan tim. spellF1 pada jendela ini 0,434 sampai 0,516 dan tidak berbeda dari 0,450 acuan 012 di dalam noise 17 possession. Perbedaan antar varian tim (88 sampai 92%) berada di dalam noise 5 menit.

**Calon saat ini:** `v3_r10_013`, dipilih atas dasar prinsip (radius dari literatur; kepala tim yang divalidasi antar pertandingan pada 013), bukan skor.

**Keterbatasan.**
- Label bola mati Metrica turunan kasar saya dari peristiwa; keselarasannya dengan definisi GT dibuktikan di sini hanya pada jendela 5 menit (recall 78%, presisi 85%), jadi akurasi pada seluruh pertandingan tidak terverifikasi.
- Model hanya dilatih pada dua pertandingan (Game 3 dikecualikan karena porsi mati labelnya hanya 10%), dan keduanya anonim sehingga kesetaraan dengan pertandingan ini tidak diketahui.
- Kenaikan macroF1 tidak sama dengan kenaikan spellF1, dan spellF1 tetap tidak teruji secara berarti pada 17 possession.

**Langkah berikutnya.** Skrip tunggal yang membangun berkas akhir dari data (syarat reproduksibilitas), uji kestabilan terhadap ambang `D` dan radius secara internal, dan dokumen teknis.